# SplatStream Lab — CUDA validation notebook

Use a **GPU runtime**. This notebook trains/evaluates an open 3D Gaussian Splatting scene with `gsplat`, then exports a PLY for the compression harness.

This is the step that converts the portable CPU validation project into a full real-scene Gaussian Splatting experiment suitable for the Netflix application.

In [ ]:
import torch, os, platform
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO CUDA')
assert torch.cuda.is_available(), 'Switch Colab to a GPU runtime first.'

## 1. Clone and install gsplat

In [ ]:
!git clone https://github.com/nerfstudio-project/gsplat.git /content/gsplat
%cd /content/gsplat
!python -m pip install -e .
!python -m pip install -r examples/requirements.txt --no-build-isolation

## 2. Download the open Mip-NeRF 360 benchmark data

The gsplat repository includes a downloader used by its own evaluation examples.

In [ ]:
%cd /content/gsplat/examples
!python datasets/download_dataset.py
!ls data/360_v2/bonsai | head

## 3. Train a 7k-step baseline on Bonsai

This is a practical first run. Keep the generated stats. For a stronger final report, repeat at 30k and on additional scenes.

The command follows gsplat's `simple_trainer.py` interface and explicitly saves a PLY.

In [ ]:
import time, pathlib, json, os
start = time.time()
!CUDA_VISIBLE_DEVICES=0 python simple_trainer.py default \
  --disable_viewer \
  --disable_video \
  --data_factor 2 \
  --data_dir data/360_v2/bonsai \
  --result_dir results/splatstream_bonsai_7k \
  --max_steps 7000 \
  --eval_steps 7000 \
  --save_steps 7000 \
  --save_ply True \
  --ply_steps 7000
elapsed = time.time() - start
print('Training wall time (s):', elapsed)

## 4. Inspect generated metrics and PLY

Exact output subpaths can evolve between gsplat versions, so the cell searches the result directory rather than assuming a single filename.

In [ ]:
from pathlib import Path
result = Path('/content/gsplat/examples/results/splatstream_bonsai_7k')
stats = sorted(result.rglob('*.json'))
plys = sorted(result.rglob('*.ply'))
print('Stats files:')
for p in stats: print(' ', p)
print('\nPLY files:')
for p in plys: print(' ', p, p.stat().st_size/1024/1024, 'MiB')

In [ ]:
for p in stats:
    try:
        print('\n###', p.name)
        print(json.dumps(json.loads(p.read_text()), indent=2)[:5000])
    except Exception as e:
        print('Could not parse', p, e)

## 5. Clone SplatStream Lab and run the compression harness

The repository is public, so Colab can clone it directly. No manual ZIP upload is required.

In [ ]:
!git clone https://github.com/reusahn/splatstream-lab.git /content/splatstream-lab
%cd /content/splatstream-lab
!pip install -e .

Select the largest PLY produced by training as the candidate scene. Then run the portable compression sweep. This step measures parameter/payload behavior; the final report should additionally use full CUDA held-out rendering for the compressed variants.

In [ ]:
plys = sorted(Path('/content/gsplat/examples/results/splatstream_bonsai_7k').rglob('*.ply'), key=lambda p: p.stat().st_size)
assert plys, 'No PLY was found. Inspect the training output above.'
PLY = plys[-1]
print('Using:', PLY)
!python -m splatstream.inspect_ply "$PLY"
!python -m splatstream.real_ply_experiment "$PLY" --output results/bonsai_portable --max-gaussians 50000

## 6. Final evidence to save

Save these before submitting:

- GPU model
- gsplat commit/version
- dataset + license/source
- training wall time
- baseline Gaussian count
- baseline PLY size
- held-out PSNR / SSIM / LPIPS from gsplat evaluation
- compression payload size / encode time
- compressed held-out quality after full CUDA rendering
- screenshots or a short orbit render

Do not use the CPU-reference PSNR as the main final quality result.